# Preparing Data for Modelling

Reads the filtered interim dataset produced by notebook `02`, imputes missing vehicle start years across the full dataset, and builds the person-interval counting-process frames consumed by the survival models.

The resulting feature datasets are saved under `Data/final`. Dataset splitting, resampling, modelling, and evaluation are performed later in R.


In [1]:
# Locate the Coding project root so the src package imports resolve consistently
import os
import polars as pl
from pathlib import Path

PROJECT_ROOT = next(
    candidate
    for parent in (Path.cwd(), *Path.cwd().parents)
    for candidate in (parent, parent / "Coding")
    if (candidate / "src").is_dir() and (candidate / "notebooks").is_dir()
)
os.chdir(PROJECT_ROOT)
PROJECT_ROOT

'c:\\Users\\Tomas\\Desktop\\Thesis Stuff\\Survival_Analysis_Thesis\\Coding'

In [2]:
from src.constants import paths_to_files_and_folders as const
from src.data_processing import DataProcessor

In [3]:
# Filtered interim files produced by the cleaning step, one per user segment
path_to_personal_filtered = const.PATH_TO_INTERIM_DATA / "personal_users_filtered.csv"
path_to_professional_filtered = const.PATH_TO_INTERIM_DATA / "professional_users_filtered.csv"

SAVE_TO_14_PERSONAL = const.PATH_TO_FINAL_DATA / "features_personal_14_day_intervals_new_features.csv" 
SAVE_TO_28_PERSONAL = const.PATH_TO_FINAL_DATA / "features_personal_28_day_intervals_new_features.csv" 

In [4]:
personal = pl.read_csv(path_to_personal_filtered)

In [5]:
# DataProcessor applies imputation and feature engineering to the complete segment dataset
processor = DataProcessor(personal)

In [6]:
personal_features_14 = processor.apply_feature_engineering(
    df=personal,
    interval_in_days=14,
    save_file_to=SAVE_TO_14_PERSONAL,
    lookback_periods=(1, 2, 3),
)

personal_features_28 = processor.apply_feature_engineering(
    df=personal,
    interval_in_days=28,
    save_file_to=SAVE_TO_28_PERSONAL,
    lookback_periods=(1,),
)


c:\Users\Tomas\Desktop\Thesis Stuff\Survival_Analysis_Thesis\Coding\src\data_processing.py:172: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  df_with_intervals = df.join_asof(
c:\Users\Tomas\Desktop\Thesis Stuff\Survival_Analysis_Thesis\Coding\src\data_processing.py:172: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  df_with_intervals = df.join_asof(


In [7]:
personal_features_14.columns

['user_id',
 'interval_start',
 'interval_end',
 'prop_clear_0_56',
 'prop_coding_0_56',
 'prop_history_screen_0_56',
 'prop_live_data_0_56',
 'prop_oca_0_56',
 'prop_scan_0_56',
 'prop_main_0_56',
 'prop_main_drift_0_28_vs_28_56',
 'n_sessions_0_28_days',
 'sessions_intensity_drift_0_28_vs_28_56_days',
 'actions_per_session_0_28_days',
 'actions_per_session_intensity_drift_0_28_vs_28_56_days',
 'recency',
 'vehicle_mean_age_overall',
 'prop_in_prod',
 'overall_prop_vehicle_make_audi',
 'overall_prop_vehicle_make_skoda',
 'overall_prop_vehicle_make_volkswagen',
 'churn_triggered_adjusted',
 'active_flag_0_56_days',
 'CV_gap_0_56_days',
 'mean_gap_0_56_days',
 'sd_gap_0_56_days']